# 04 — Visualisasi Final (6 Figure)
**Input:** `Data/topsis_result.csv` · `Data/clustered_result.csv` · `Data/interim/{kecamatan_poly.geojson, cluster_features_z.csv, kmeans_sweep.csv}`
**Output:** `Data/figures/final/fig0{1,3,4,5,6,7}_*.png` (300 DPI, caption Bahasa Indonesia)

Semua angka pada judul/caption dihitung dari data, bukan ditulis tangan. Skor TOPSIS dihitung ulang dengan `src/topsis.py` dan harus identik dengan `topsis_result.csv` (assert di bagian setup).

| Fig | Section | Konten |
|---|---|---|
| 01 | Metode | Peta cluster |
| 03 | Pembobotan | 3 skema bobot (8 kriteria) |
| 04 | Hasil | Ranking 42 kecamatan |
| 05 | Evaluasi | Pemilihan k |
| 06 | Profil | PCA + centroid |
| 07 | Validasi | Sensitivitas bobot (1000 simulasi) |

## 0. Setup

In [ ]:
import sys, warnings, os
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as mpatches, matplotlib.patheffects as pe
from matplotlib.lines import Line2D
import seaborn as sns, geopandas as gpd
from sklearn.decomposition import PCA
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")

ROOT = Path("..").resolve(); sys.path.insert(0, str(ROOT))
from src.topsis import shannon_entropy_weights, topsis

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11, "axes.titleweight": "bold", "axes.titlesize": 13,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25, "grid.linestyle": "--",
    "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": .15, "figure.facecolor": "white", "axes.facecolor": "white"})

PALETTE = {"Prioritas Tinggi": "#8B0000", "Waspada": "#E4572E", "Menengah": "#F6C85F", "Rendah": "#B7E4A7"}   # sama dengan 03_kmeans
LABEL_ORDER = list(PALETTE); NOTE = "#64748b"; EDGE = "#cbd5e1"
DATA = ROOT/"Data"; FIG = DATA/"figures"/"final"; FIG.mkdir(parents=True, exist_ok=True)
print("Output dir:", FIG)

In [ ]:
topsis_df = pd.read_csv(DATA/"topsis_result.csv"); topsis_df["kecamatan"] = topsis_df.kecamatan.str.strip().str.upper(); topsis_df = topsis_df.set_index("kecamatan")
clustered = pd.read_csv(DATA/"clustered_result.csv").set_index("kecamatan")
poly = gpd.read_file(DATA/"interim/kecamatan_poly.geojson")
Z = pd.read_csv(DATA/"interim/cluster_features_z.csv", index_col=0); sweep = pd.read_csv(DATA/"interim/kmeans_sweep.csv", index_col="k")

# kriteria TOPSIS final (sama dengan 02_topsis.ipynb): 7 benefit + 1 cost; bobot pakar dari 02
CRIT = {"frek_tahun_banjir": ("benefit", .10, "Frekuensi\ntahun banjir"), "bulan_banjir_per_tahun": ("benefit", .15, "Bulan banjir\n/tahun"),
        "kejadian_per_tahun": ("benefit", .15, "Kejadian\n/tahun"), "tinggi_max_cm": ("benefit", .10, "Tinggi maks\ngenangan"),
        "lama_hari_mean": ("benefit", .10, "Lama\ngenangan"), "jiwa_per_1000": ("benefit", .20, "Jiwa terdampak\n/1000"),
        "kepadatan_jiwa_km2": ("benefit", .10, "Kepadatan\npenduduk"), "n_alat_dini": ("cost", .10, "Alat peringatan\ndini (cost)")}
cols = list(CRIT); types = [CRIT[c][0] for c in cols]
Xm = topsis_df[cols].values.astype(float)
w_entropy = shannon_entropy_weights(Xm); w_equal = np.full(len(cols), 1/len(cols)); w_expert = np.array([CRIT[c][1] for c in cols]); w_expert /= w_expert.sum()
score = topsis(Xm, w_entropy, types)[0]
assert np.allclose(score, topsis_df.skor_topsis.values, atol=1e-9), "skor TOPSIS hitung ulang != topsis_result.csv"
assert set(clustered.index) == set(topsis_df.index) == set(Z.index)
RANK = topsis_df.rank_topsis; TOP5 = RANK.nsmallest(5).index.tolist()
print("OK: TOPSIS direproduksi persis |", len(topsis_df), "kecamatan | top-5:", [k.title() for k in TOP5])
poly_c = poly.merge(clustered.reset_index()[["kecamatan", "cluster_name"]], on="kecamatan", how="left")

## Fig 1 — Peta Cluster `[Metode]`

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
for lab in LABEL_ORDER: poly_c[poly_c.cluster_name == lab].plot(ax=ax, color=PALETTE[lab], edgecolor="white", linewidth=.9)
poly_c[poly_c.cluster_name.isna()].plot(ax=ax, color="#e5e7eb", edgecolor="white", linewidth=.9)
halo = [pe.withStroke(linewidth=3, foreground="#1e293b")]
for kec in TOP5:
    g = poly_c[poly_c.kecamatan == kec].geometry.iloc[0].representative_point()
    ax.annotate(f"#{int(RANK[kec])} {kec.title()}", (g.x, g.y), fontsize=9, fontweight="bold", ha="center", color="white", path_effects=halo)
n_per = clustered.cluster_name.value_counts()
handles = [mpatches.Patch(facecolor=PALETTE[l], edgecolor="white", label=f"{l} (n={n_per.get(l, 0)})") for l in LABEL_ORDER] + [mpatches.Patch(facecolor="#e5e7eb", edgecolor="white", label="Tidak ada data")]
ax.legend(handles=handles, loc="lower left", frameon=True, fancybox=False, edgecolor=EDGE, title="Tingkat prioritas", fontsize=9, title_fontsize=10)
ax.set_axis_off(); ax.set_title("Distribusi Prioritas Mitigasi Banjir 42 Kecamatan DKI Jakarta\n(K-Means k=4 atas skor TOPSIS, dampak warga, dan kepadatan)", fontsize=13, pad=14, loc="left")
fig.text(.5, .04, "Sumber: Satu Data Jakarta 2013–2024 · Model: TOPSIS (bobot entropi) + K-Means · Label #n = peringkat TOPSIS", ha="center", fontsize=8, style="italic", color=NOTE)
plt.savefig(FIG/"fig01_peta_cluster.png"); plt.show()

## Fig 3 — Perbandingan 3 Skema Bobot `[Pembobotan]`

In [ ]:
bobot = pd.DataFrame({"Equal": w_equal, "Entropy": w_entropy, "Expert": w_expert}, index=[CRIT[c][2] for c in cols])
fig, ax = plt.subplots(figsize=(13, 5.8)); x = np.arange(len(bobot)); w = .27
cs = {"Equal": "#94a3b8", "Entropy": "#0369a1", "Expert": "#f59e0b"}
for i, s in enumerate(bobot.columns):
    bars = ax.bar(x + (i-1)*w, bobot[s], w, label=s, color=cs[s], edgecolor="white", linewidth=.8)
    if s == "Entropy":
        for b_, v in zip(bars, bobot[s]): ax.text(b_.get_x()+b_.get_width()/2, v+.008, f"{v:.2f}", ha="center", fontsize=9, fontweight="bold", color=cs[s])
ax.set_xticks(x); ax.set_xticklabels(bobot.index, fontsize=9); ax.set_ylabel("Bobot kriteria"); ax.set_ylim(0, bobot.values.max()*1.18)
ax.set_title("Perbandingan 3 Skema Pembobotan Kriteria TOPSIS (8 kriteria)\nEntropy dipakai sebagai bobot final", loc="left"); ax.legend(frameon=True, edgecolor=EDGE)
top_c = bobot.Entropy.idxmax().replace("\n", " ")
fig.text(.5, -.03, f"Bobot entropi tertinggi: {top_c} ({bobot.Entropy.max():.2f}). Kriteria dengan sebaran antar-kecamatan paling tidak merata mendapat bobot terbesar.", ha="center", fontsize=8, style="italic", color=NOTE)
plt.tight_layout(); plt.savefig(FIG/"fig03_bobot_kriteria.png"); plt.show(); display(bobot.round(3))

## Fig 4 — Peringkat 42 Kecamatan `[Hasil]`

In [ ]:
df = clustered.sort_values("skor_topsis"); fig, ax = plt.subplots(figsize=(10, 12))
bars = ax.barh(df.index.str.title(), df.skor_topsis, color=df.cluster_name.map(PALETTE), edgecolor="white", linewidth=.6, height=.75)
for b_, v in zip(bars, df.skor_topsis): ax.text(v+.008, b_.get_y()+b_.get_height()/2, f"{v:.3f}", va="center", fontsize=8, color="#334155")
t5 = [k.title() for k in TOP5]
for l in ax.get_yticklabels():
    if l.get_text() in t5: l.set_fontweight("bold")
ax.set_xlabel("Skor preferensi TOPSIS Ci (0 = rendah, 1 = tinggi)"); ax.set_xlim(0, 1)
ax.set_title("Peringkat 42 Kecamatan DKI Jakarta berdasarkan Skor TOPSIS\n(bobot entropi, 8 kriteria)", loc="left")
ax.tick_params(axis="y", labelsize=9); ax.grid(axis="y", visible=False)
ax.legend(handles=[mpatches.Patch(color=PALETTE[l], label=l) for l in LABEL_ORDER], loc="lower right", frameon=True, edgecolor=EDGE, title="Cluster", fontsize=9)
hi, lo = df.index[-1].title(), df.index[0].title()
fig.text(.5, -.005, f"Teks tebal = Top-5. Peringkat 1: {hi} ({df.skor_topsis.iloc[-1]:.3f}); peringkat {len(df)}: {lo} ({df.skor_topsis.iloc[0]:.3f}).", ha="center", fontsize=8, style="italic", color=NOTE)
plt.tight_layout(); plt.savefig(FIG/"fig04_ranking_42kec.png"); plt.show()

## Fig 5 — Pemilihan k `[Evaluasi]`

In [ ]:
ok = (sweep.ukuran_min >= 5) & (sweep.ukuran_maks <= .40*len(Z)) & (sweep.spearman_rank >= .80)    # aturan kelayakan dari 03_kmeans
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))
for ax, (col, title, hint) in zip(axes, [("inertia", "Elbow (Inertia)", "lebih kecil = lebih baik"), ("silhouette", "Silhouette", "lebih besar = lebih baik"),
                                           ("DBI", "Davies-Bouldin", "lebih kecil = lebih baik"), ("CH", "Calinski-Harabasz", "lebih besar = lebih baik")]):
    for k in sweep.index[ok]: ax.axvspan(k-.4, k+.4, color="#e0f2fe", zorder=0)
    ax.plot(sweep.index, sweep[col], marker="o", color="#0369a1", lw=2, ms=8, mec="white", mew=1.5, zorder=3)
    ax.axvline(4, color="#8B0000", ls="--", lw=1.2, alpha=.7); ax.scatter([4], [sweep.loc[4, col]], s=150, color="#8B0000", zorder=4, edgecolor="white")
    ax.set_title(title, fontsize=12); ax.set_xlabel("Jumlah cluster (k)"); ax.set_xticks(sweep.index); ax.text(.5, -.25, hint, transform=ax.transAxes, ha="center", fontsize=8, style="italic", color=NOTE)
best = sweep.silhouette[ok].idxmax()
fig.suptitle("Evaluasi Pemilihan Jumlah Cluster", fontsize=14, fontweight="bold", y=1.03)
fig.text(.5, -.06, f"Area biru = k yang lolos aturan kelayakan (ukuran cluster 5–40% data, Spearman cluster–rank ≥ 0,8). k=4 dipilih sebagai 4 tingkat prioritas kebijakan "
         f"(silhouette {sweep.loc[4,'silhouette']:.3f}; terbaik di antara k layak: k={best}, {sweep.loc[best,'silhouette']:.3f}).", ha="center", fontsize=8, style="italic", color=NOTE)
plt.tight_layout(); plt.savefig(FIG/"fig05_pilih_k_validity.png"); plt.show(); display(sweep.round(3))

## Fig 6 — PCA & Profil Centroid `[Profil]`

In [ ]:
lab = clustered.reindex(Z.index).cluster_id.values; pca = PCA(2, random_state=42).fit(Z); pc = pca.transform(Z)
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw={"width_ratios": [1.2, 1]})
ax = axes[0]
for i, l in enumerate(LABEL_ORDER):
    m_ = lab == i; ax.scatter(pc[m_, 0], pc[m_, 1], color=PALETTE[l], s=140, edgecolor="white", lw=1.2, alpha=.95, label=f"{l} (n={m_.sum()})", zorder=3)
for kec in TOP5:
    j = list(Z.index).index(kec); ax.annotate(kec.title(), (pc[j, 0], pc[j, 1]), fontsize=9, fontweight="bold", xytext=(5, 5), textcoords="offset points", color="#1e293b")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} variansi)"); ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} variansi)")
ax.set_title(f"Separasi 4 Cluster di Ruang PCA\n(variansi dijelaskan: {pca.explained_variance_ratio_.sum():.0%})", loc="left")
ax.axhline(0, color=EDGE, lw=.8, zorder=1); ax.axvline(0, color=EDGE, lw=.8, zorder=1); ax.legend(frameon=True, edgecolor=EDGE, fontsize=9, title="Cluster")
ax = axes[1]; nice = {"skor_topsis": "Skor\nTOPSIS", "jiwa_per_1000": "Jiwa terdampak\n/1000", "kepadatan_jiwa_km2": "Kepadatan\npenduduk"}
cen = Z.groupby(lab).mean(); cen.index = LABEL_ORDER[:len(cen)]; cen.columns = [nice.get(c, c) for c in cen.columns]
sns.heatmap(cen, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-2, vmax=2, cbar_kws={"label": "z-score centroid", "shrink": .75}, linewidths=.8, linecolor="white", ax=ax, annot_kws={"fontsize": 10, "fontweight": "bold"})
ax.set_title("Profil Centroid per Cluster (z-score)", loc="left"); ax.set_xlabel(""); ax.set_ylabel(""); plt.setp(ax.get_yticklabels(), rotation=0, fontweight="bold"); plt.setp(ax.get_xticklabels(), rotation=0)
ax.grid(False); plt.tight_layout(); plt.savefig(FIG/"fig06_pca_centroid.png"); plt.show()

## Fig 7 — Sensitivitas Bobot `[Validasi]`

In [ ]:
rng = np.random.default_rng(42); NSIM = 1000
rk = lambda s: pd.Series(s, index=topsis_df.index).rank(ascending=False, method="min")
RK_base = rk(score); sims = []
for _ in range(NSIM):
    w_ = w_entropy * rng.uniform(.7, 1.3, len(w_entropy)); w_ /= w_.sum(); sims.append(rk(topsis(Xm, w_, types)[0]))
sims = pd.DataFrame(sims); rhos = np.array([spearmanr(RK_base, r)[0] for _, r in sims.iterrows()])
t10 = set(RK_base.nsmallest(10).index); keep = np.mean([len(set(r.nsmallest(10).index) & t10)/10 for _, r in sims.iterrows()])
t5s = set(TOP5); keep5 = np.mean([len(set(r.nsmallest(5).index) & t5s)/5 for _, r in sims.iterrows()])

fig, axes = plt.subplots(1, 2, figsize=(16, 10), gridspec_kw={"width_ratios": [2.2, 1]}); ax = axes[0]
order = RK_base.sort_values().index.tolist()
bp = ax.boxplot([sims[k].values for k in order], vert=False, patch_artist=True, tick_labels=[k.title() for k in order], widths=.7, showfliers=False,
                medianprops=dict(color="white", lw=1.5), boxprops=dict(lw=.5), whiskerprops=dict(lw=.8))
for patch, k in zip(bp["boxes"], order): patch.set_facecolor(PALETTE[clustered.loc[k, "cluster_name"]]); patch.set_alpha(.9)
for i, k in enumerate(order): ax.scatter(RK_base[k], i+1, color="black", marker="D", s=25, zorder=5, edgecolor="white", lw=.8)
ax.set_xlabel("Peringkat (1 = prioritas tertinggi)"); ax.set_xlim(0, 43); ax.invert_yaxis(); ax.tick_params(axis="y", labelsize=8); ax.grid(axis="y", visible=False)
ax.set_title(f"Stabilitas Peringkat: Distribusi Rank per Kecamatan\n({NSIM} simulasi, bobot entropi diganggu ±30%)", loc="left")
ax.legend(handles=[mpatches.Patch(color=PALETTE[l], label=l, alpha=.9) for l in LABEL_ORDER] + [Line2D([0], [0], marker="D", color="black", ls="None", label="Peringkat dasar", ms=7, mec="white")], loc="upper right", frameon=True, edgecolor=EDGE, fontsize=8)
ax = axes[1]; ax.hist(rhos, bins=30, color="#0369a1", edgecolor="white", lw=.8)
ax.axvline(rhos.mean(), color="#8B0000", ls="--", lw=2, label=f"Rata-rata: {rhos.mean():.3f}"); ax.axvline(rhos.min(), color="#f59e0b", ls=":", lw=1.5, label=f"Minimum: {rhos.min():.3f}")
ax.set_xlabel("Spearman vs peringkat dasar"); ax.set_ylabel(f"Frekuensi (dari {NSIM} simulasi)"); ax.set_title("Korelasi Peringkat vs Dasar", loc="left"); ax.legend(frameon=True, edgecolor=EDGE, fontsize=9, loc="upper left", bbox_to_anchor=(0, .86))
ax.text(.02, .98, f"Top-10 bertahan: {keep:.0%}\nTop-5 bertahan: {keep5:.0%}", transform=ax.transAxes, fontsize=10, fontweight="bold", va="top", bbox=dict(facecolor="#e0f2fe", edgecolor="#0369a1", boxstyle="round,pad=.5"))
plt.tight_layout(); plt.savefig(FIG/"fig07_sensitivity.png"); plt.show()
print(f"Spearman rata-rata {rhos.mean():.3f} (min {rhos.min():.3f}) | Top-10 bertahan {keep:.1%} | Top-5 bertahan {keep5:.1%}")

In [ ]:
for f in sorted(FIG.glob("fig0*.png")): print(f"{f.name:34s} {os.path.getsize(f)/1024:7.1f} KB")